Clonar o repositorio

In [1]:
!git clone https://github.com/carbon-footprint-analysis/carbon-footprint-analysis.git

Cloning into 'carbon-footprint-analysis'...
remote: Enumerating objects: 476, done.
remote: Counting objects: 100% (214/214), done.
remote: Compressing objects: 100% (127/127), done.
remote: Total 476 (delta 113), reused 173 (delta 86), pack-reused 262 (from 1)
Receiving objects: 100% (476/476), 31.88 MiB | 6.93 MiB/s, done.
Resolving deltas: 100% (200/200), done.


Instalar a versão especifica para o projeto (VsCode x Colab)

In [2]:
pip install scikit-learn==1.8.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 60.7 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


Importar as bibliotecas

In [3]:
import joblib
import urllib.request
import pandas as pd

Importando o arquivo joblib

In [4]:
model = joblib.load("/content/carbon-footprint-analysis/models/best_carbon_footprint_model.joblib")

Analisando o modelo para sanity

In [5]:
print(model)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['energy_kwh', 'month']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['state', 'usage_type',
                                                   'energy_source',
                                                   'season'])])),
                ('regressor',
                 GradientBoostingRegressor(max_depth=5, random_state=42))])


In [6]:
encoder = model.named_steps['preprocessor'].named_transformers_['cat']

encoder.categories_

[array(['AC', 'AL', 'AM', 'AP', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MG',
        'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN', 'RO', 'RR',
        'RS', 'SC', 'SE', 'SP', 'TO'], dtype=object),
 array(['comercial', 'industrial', 'outros', 'residencial', 'rural'],
       dtype=object),
 array(['hydro', 'nuclear', 'solar', 'thermal', 'wind'], dtype=object),
 array(['Inverno', 'Outono', 'Primavera', 'Verao'], dtype=object)]

Testando o modelo

In [7]:
df = pd.DataFrame({
    "energy_kwh": [1200],
    "month": [6],
    "state": ["SP"],
    "usage_type": ["industrial"],
    "energy_source": ["hydro"],
    "season": ["Inverno"]
})

model.predict(df)

array([23.18777434])

Fazendo o Wrapper


Utilizando o modelo para fazer o comparativo entre todas as fontes de energia eletrica

In [8]:
def predict_all_sources(model, energy_kwh, month, state, usage_type, season):
    sources = ['hydro', 'nuclear', 'solar', 'thermal', 'wind']

    results = {}

    for source in sources:

        df = pd.DataFrame({
            "energy_kwh": [energy_kwh],
            "month": [month],
            "state": [state],
            "usage_type": [usage_type],
            "energy_source": [source],
            "season": [season]
        })

        co2 = model.predict(df)[0]

        results[source] = round(float(co2), 2)

    results_sorted = dict(sorted(results.items(), key=lambda x: x[1]))

    return results_sorted


In [9]:
predict_all_sources(
    model,
    energy_kwh=1200,
    month=6,
    state="SP",
    usage_type="industrial",
    season="Inverno"
)

{'wind': 13.08,
 'nuclear': 17.84,
 'hydro': 23.19,
 'solar': 23.19,
 'thermal': 724.81}

Criando comparativo para combustiveis liquidos foi utilizado a media de motores motores modernos. E os valores foram retirados do `PCC Guidelines for National Greenhouse Gas Inventories`, `IEA – International Energy Agency` e `US EPA emission factors`

In [10]:
def liquid_fuel_emissions(energy_kwh):

    fuels = {
        "ethanol": {
            "efficiency": 0.27,
            "emission_factor": 0.20
        },
        "gasoline": {
            "efficiency": 0.30,
            "emission_factor": 0.64
        },
        "diesel": {
            "efficiency": 0.38,
            "emission_factor": 0.73
        }
    }

    results = {}

    for fuel, data in fuels.items():

        efficiency = data["efficiency"]
        factor = data["emission_factor"]

        energy_input = energy_kwh / efficiency
        co2 = energy_input * factor

        results[fuel] = round(co2, 2)

    results_sorted = dict(sorted(results.items(), key=lambda x: x[1]))

    return results_sorted

In [11]:
liquid_fuel_emissions(1200)

{'ethanol': 888.89, 'diesel': 2305.26, 'gasoline': 2560.0}

Unificando fontes geradoras e ordenando do menos poluente para o mais poluente adicionado a % de emissão de Co2 que foi aumentado ou diminuido se comparado com a fonte geradora base do brasil `Hydro`

In [12]:
def compare_energy_sources(model, energy_kwh, month, state, usage_type, season):

    electricity = predict_all_sources(
        model,
        energy_kwh,
        month,
        state,
        usage_type,
        season
    )

    fuels = liquid_fuel_emissions(energy_kwh)

    combined = {**electricity, **fuels}

    ranking = dict(sorted(combined.items(), key=lambda x: x[1]))

    base = ranking["hydro"]

    result = {}

    for source, value in ranking.items():

        percent = ((value - base) / base) * 100

        result[source] = {
            "co2": round(value, 2),
            "vs_hydro_%": round(percent, 2)
        }

    return result

In [13]:
compare_energy_sources(
    model,
    energy_kwh=1200,
    month=6,
    state="SP",
    usage_type="industrial",
    season="Inverno"
)

{'wind': {'co2': 13.08, 'vs_hydro_%': -43.6},
 'nuclear': {'co2': 17.84, 'vs_hydro_%': -23.07},
 'hydro': {'co2': 23.19, 'vs_hydro_%': 0.0},
 'solar': {'co2': 23.19, 'vs_hydro_%': 0.0},
 'thermal': {'co2': 724.81, 'vs_hydro_%': 3025.53},
 'ethanol': {'co2': 888.89, 'vs_hydro_%': 3733.07},
 'diesel': {'co2': 2305.26, 'vs_hydro_%': 9840.75},
 'gasoline': {'co2': 2560.0, 'vs_hydro_%': 10939.24}}

Versão Final Wrapper

In [1]:
!pip install scikit-learn==1.8.0

!git clone https://github.com/carbon-footprint-analysis/carbon-footprint-analysis.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 49.5 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
Cloning into 'carbon-footprint-analysis'...
remote: Enumerating objects: 476, done.
remote: Counting objects: 100% (214/214), done.
remote: Compressing objects: 100% (127/127), done.
remote: Total 476 (delta 113), reused 173 (delta 86), pack-reused 262 (from 1)
Receiving objects: 100% (476/476), 31.88 MiB | 21.85 MiB/s, done.
Resolving deltas: 100% (200/200), done.


Criando .py do wrapper

In [2]:
%%writefile wrapper.py
import joblib
import pandas as pd

model = joblib.load("/content/carbon-footprint-analysis/models/best_carbon_footprint_model.joblib")


def predict_all_sources(model, energy_kwh, month, state, usage_type, season):

    sources = ['hydro', 'nuclear', 'solar', 'thermal', 'wind']
    results = {}

    for source in sources:

        df = pd.DataFrame({
            "energy_kwh": [energy_kwh],
            "month": [month],
            "state": [state],
            "usage_type": [usage_type],
            "energy_source": [source],
            "season": [season]
        })

        co2 = model.predict(df)[0]

        results[source] = round(float(co2), 2)

    return dict(sorted(results.items(), key=lambda x: x[1]))


def liquid_fuel_emissions(energy_kwh):

    fuels = {
        "ethanol": {"efficiency": 0.27, "emission_factor": 0.20},
        "gasoline": {"efficiency": 0.30, "emission_factor": 0.64},
        "diesel": {"efficiency": 0.38, "emission_factor": 0.73}
    }

    results = {}

    for fuel, data in fuels.items():

        energy_input = energy_kwh / data["efficiency"]
        co2 = energy_input * data["emission_factor"]

        results[fuel] = round(co2, 2)

    return dict(sorted(results.items(), key=lambda x: x[1]))


def compare_energy_sources(energy_kwh, month, state, usage_type, season):

    electricity = predict_all_sources(
        model,
        energy_kwh,
        month,
        state,
        usage_type,
        season
    )

    fuels = liquid_fuel_emissions(energy_kwh)

    combined = {**electricity, **fuels}

    ranking = dict(sorted(combined.items(), key=lambda x: x[1]))

    return ranking

Writing wrapper.py


In [3]:
!ls

carbon-footprint-analysis  sample_data	wrapper.py


importando o wrapper diretamente do .py para possivel implementação no FastAPI

In [4]:
import importlib
import wrapper

importlib.reload(wrapper)

<module 'wrapper' from '/content/wrapper.py'>

Diversos testes para verificar confiabilidade do wrapper

In [5]:
print(wrapper.compare_energy_sources(1200, 6, "SP", "industrial", "Inverno"))

print(wrapper.compare_energy_sources(800, 3, "RJ", "commercial", "Outono"))

print(wrapper.compare_energy_sources(1500, 12, "MG", "residencial", "Verão"))

print(wrapper.compare_energy_sources(600, 9, "RS", "industrial", "Primavera"))

print(wrapper.compare_energy_sources(2000, 1, "BA", "commercial", "Verão"))

print(wrapper.compare_energy_sources(400, 7, "SC", "residencial", "Inverno"))

print(wrapper.compare_energy_sources(1000, 5, "GO", "rural", "Outono"))

print(wrapper.compare_energy_sources(1800, 10, "PA", "industrial", "Primavera"))

print(wrapper.compare_energy_sources(500, 4, "CE", "commercial", "Outono"))

print(wrapper.compare_energy_sources(2200, 8, "PR", "industrial", "Inverno"))

{'wind': 13.08, 'nuclear': 17.84, 'hydro': 23.19, 'solar': 23.19, 'thermal': 724.81, 'ethanol': 888.89, 'diesel': 2305.26, 'gasoline': 2560.0}
{'wind': 9.84, 'nuclear': 12.04, 'hydro': 17.23, 'solar': 17.23, 'thermal': 464.72, 'ethanol': 592.59, 'diesel': 1536.84, 'gasoline': 1706.67}
{'wind': 18.4, 'nuclear': 19.34, 'hydro': 30.26, 'solar': 30.26, 'thermal': 902.47, 'ethanol': 1111.11, 'diesel': 2881.58, 'gasoline': 3200.0}
{'wind': 7.04, 'nuclear': 7.94, 'hydro': 13.13, 'solar': 13.13, 'thermal': 352.35, 'ethanol': 444.44, 'diesel': 1152.63, 'gasoline': 1280.0}
{'wind': 24.98, 'nuclear': 27.97, 'hydro': 41.33, 'solar': 41.33, 'thermal': 1228.21, 'ethanol': 1481.48, 'diesel': 3842.11, 'gasoline': 4266.67}
{'nuclear': 4.06, 'wind': 5.14, 'hydro': 9.26, 'solar': 9.26, 'thermal': 233.25, 'ethanol': 296.3, 'diesel': 768.42, 'gasoline': 853.33}
{'wind': 11.87, 'nuclear': 15.17, 'hydro': 20.36, 'solar': 20.36, 'thermal': 618.62, 'ethanol': 740.74, 'diesel': 1921.05, 'gasoline': 2133.33}
{'w